# Time Series Analysis Exercises

## Exercise 0: Environment and libraries
The goal of this exercise is to set up the Python work environment with the required libraries.

- Python >= 3.9
- pandas
- numpy
- jupyter
- plotly

We assume these libraries are installed in the current environment.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

## Exercise 1: Series
The goal of this exercise is to learn to manipulate time series in Pandas.

1. Create a Series named `integer_series` from 1st January 2010 to 31 December 2020. At each date is associated the number of days since 1st January 2010. It starts with 0.
2. Using Pandas, compute a 7 days moving average. This transformation smooths the time series by removing small fluctuations. **without for loop**.

In [ ]:
# 1. Create the Series
dates = pd.date_range(start='2010-01-01', end='2020-12-31', freq='D')
values = np.arange(len(dates))
integer_series = pd.Series(data=values, index=dates, name='integer_series')

print(integer_series.head())
print(integer_series.tail())

In [ ]:
# 2. Compute 7-day moving average
moving_average = integer_series.rolling(window=7).mean()

print(moving_average.head(10))

## Exercise 2: Financial data
This exercise aims to familiarize you with handling financial data using Pandas.

The data we will use is Apple stock.

1. Preprocess: check for missing values, convert string dates to datetime objects, and set the date column as the index.
2. Use Plotly to generate a candlestick chart (Open, High, Low, Close).
3. Aggregate the data to last business day of each month.
4. Compute daily returns based on the Open price without using a for loop.

In [ ]:
# Load data
try:
    df = pd.read_csv('../data/AAPL.csv')
except FileNotFoundError:
    # Fallback if running from notebooks dir directly or root
    try:
        df = pd.read_csv('data/AAPL.csv')
    except FileNotFoundError:
        print("Data file not found. Please run scripts/download_data.py first.")

# 1. Preprocess
# Check for missing values
print("Missing values:\n", df.isnull().sum())

# Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Set Date as index
df.set_index('Date', inplace=True)

# Sort index just in case
df.sort_index(inplace=True)

print(df.head())

In [ ]:
# 2. Plotly Candlestick Chart
fig = go.Figure(data=[go.Candlestick(x=df.index,
                open=df['Open'],
                high=df['High'],
                low=df['Low'],
                close=df['Close'])])

fig.update_layout(title='Apple Stock Price', yaxis_title='Price')
fig.show()

In [ ]:
# 3. Aggregate to last business day of each month
# Audit requirement: Prices -> mean, Low -> min, High -> max, Volume -> sum
agg_dict = {
    'Open': 'mean',
    'High': 'max',
    'Low': 'min',
    'Close': 'mean',
    'Adj Close': 'mean',
    'Volume': 'sum'
}

monthly_df = df.resample('BM').agg(agg_dict)

print("Number of months:", len(monthly_df))
print(monthly_df.head())

In [ ]:
# 4. Compute daily returns based on Open price
# Method 1: pct_change
df['Open_Return'] = df['Open'].pct_change()

print(df[['Open', 'Open_Return']].head())

## Exercise 3: Multi asset returns
The goal of this exercise is to learn to compute daily returns on a DataFrame that contains many assets.

Without using a for loop, compute the daily returns for all the companies and returns a DataFrame.

In [ ]:
business_dates = pd.bdate_range('2021-01-01', '2021-12-31')

# Generate tickers
tickers = ['AAPL', 'FB', 'GE', 'AMZN', 'DAI']

# Create index
index = pd.MultiIndex.from_product([business_dates, tickers], names=['Date', 'Ticker'])

# Create DF
market_data = pd.DataFrame(index=index,
                        data=np.random.randn(len(index), 1),
                        columns=['Price'])

# Use pivot_table as requested by audit
pivoted_data = market_data.pivot_table(values="Price", index="Date", columns="Ticker")

# Compute daily returns
returns_df = pivoted_data.pct_change()

print(returns_df.head())

## Exercise 4: Backtest
We will backtest a long only strategy on Apple Inc. 

1. Drop rows with missing values and compute daily future return on Adjusted Close price.
2. Create a Series with random boolean array (p=0.5) for signals.
3. Backtest the signal: Buy 1$ if Signal=1 at d, sell at d+1. Profit = PnL / Invested.
   - PnL for day d: affected to day d.
4. Compute strategy return: (Total Earned - Total Invested) / Total Invested.
5. Compute "Always Buy" strategy PnL and total PnL.
6. Plot daily PnL of both.

In [ ]:
# 1. Drop missing and compute future return
# Audit instruction implies: compute return then use it. Or using full index for signals.
# Let's keep the full DF structure to maintain index integrity for signals

df_backtest = df.copy()

# Future return: Return(t) = (Price(t+1) - Price(t)) / Price(t)
df_backtest['daily_return'] = df_backtest['Adj Close'].pct_change()
df_backtest['future_return'] = (df_backtest['Adj Close'].shift(-1) - df_backtest['Adj Close']) / df_backtest['Adj Close']

# Valid rows for checking PnL are those where future_return is not NaN
# This effectively drops the last row for computation purposes later
print(df_backtest[['Adj Close', 'future_return']].head())

In [ ]:
# 2. Create random signal
np.random.seed(42) # For reproducibility
# Use randint to match audit preference
signals = np.random.randint(0, 2, len(df_backtest))
df_backtest['long_only_signal'] = signals

print(df_backtest['long_only_signal'].head())

In [ ]:
# 3. Backtest signal
# If Signal=1, PnL = future_return * 1$
# If Signal=0, PnL = 0
df_backtest['strategy_pnl'] = df_backtest['long_only_signal'] * df_backtest['future_return']

print(df_backtest[['long_only_signal', 'future_return', 'strategy_pnl']].head())

In [ ]:
# 4. Compute strategy return
# Total Invested is the sum of days where we invested (signal=1)
# Only count invested on days where PnL is possible (i.e. future_return is not NaN)
# This is a subtle point. If we invest on the last day, we can't sell tomorrow.
# So we should drop the last day before summing invested.

valid_mask = df_backtest['future_return'].notna()
df_valid = df_backtest[valid_mask].copy()

total_invested_strategy = df_valid['long_only_signal'].sum()
total_pnl_strategy = df_valid['strategy_pnl'].sum()

strategy_return = total_pnl_strategy / total_invested_strategy

print(f"Total Invested: ${total_invested_strategy}")
print(f"Total PnL: ${total_pnl_strategy:.2f}")
print(f"Strategy Return: {strategy_return:.2%}")

In [ ]:
# 5. Always Buy Strategy
# We update main DF first
df_backtest['always_buy_signal'] = 1
df_backtest['always_buy_pnl'] = df_backtest['always_buy_signal'] * df_backtest['future_return']

# Recalculate stats using the valid mask on the main DF to ensure we access the new column
total_invested_always = len(df_backtest[valid_mask])
total_pnl_always = df_backtest.loc[valid_mask, 'always_buy_pnl'].sum()
always_buy_return = total_pnl_always / total_invested_always

print(f"Total Invested (Always Buy): ${total_invested_always}")
print(f"Total PnL (Always Buy): ${total_pnl_always:.2f}")
print(f"Always Buy Return: {always_buy_return:.2%}")

In [ ]:
# 6. Plot Daily PnL
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_backtest.index, y=df_backtest['strategy_pnl'], mode='lines', name='Random Strategy PnL'))
fig.add_trace(go.Scatter(x=df_backtest.index, y=df_backtest['always_buy_pnl'], mode='lines', name='Always Buy PnL'))

fig.update_layout(title='Daily PnL: Random Strategy vs Always Buy', yaxis_title='PnL ($)')
fig.show()